In [14]:
from netgen.occ import *
from netgen.webgui import Draw as DrawGeo
import ngsolve

In [15]:
box = Box((0,0,0), (3,0.6,1))
box.faces.name="outer"
cyl = sum( [Cylinder((0.5+i,0,0.5), Y, 0.25,0.8) for i in range(3)] )
cyl.faces.name="cyl"
geo = box-cyl

DrawGeo(geo, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3,…

BaseWebGuiScene

In [16]:
cylboxedges = geo.faces["outer"].edges * geo.faces["cyl"].edges
cylboxedges.name = "cylbox"
geo = geo.MakeChamfer(cylboxedges, 0.03)

In [17]:
geo.faces.Min(X).name = "fix"
geo.faces.Max(X).name = "force"

DrawGeo(geo, height="3vh");

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3,…

In [31]:
from ngsolve import *
from ngsolve.webgui import Draw
mesh = Mesh(OCCGeometry(geo).GenerateMesh(maxh=0.1)).Curve(3)
Draw (mesh, height="3vh");

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

In [19]:
E, nu = 210, 0.2
mu  = E / 2 / (1+nu)
lam = E * nu / ((1+nu)*(1-2*nu))

def Stress(strain):
    return 2*mu*strain + lam*Trace(strain)*Id(3)

In [20]:
fes = VectorH1(mesh, order=3, dirichlet="fix")
u,v = fes.TnT()
gfu = GridFunction(fes)

with TaskManager():
    a = BilinearForm(InnerProduct(Stress(Sym(Grad(u))), Sym(Grad(v))).Compile()*dx)
    pre = Preconditioner(a, "bddc")
    a.Assemble()

In [26]:
force = CF( (1e-3,1e-6,0) )
f = LinearForm(force*v*ds("force")).Assemble()

In [27]:
from ngsolve.krylovspace import CGSolver
inv = CGSolver(a.mat, pre, printrates=True, tol=1e-8)
gfu.vec.data = inv * f.vec

CG iteration 1, residual = 0.00018037260149009163     
CG iteration 2, residual = 7.519816148112285e-05     
CG iteration 3, residual = 8.803204473852018e-05     
CG iteration 4, residual = 6.780346420996131e-05     
CG iteration 5, residual = 6.521881903334944e-05     
CG iteration 6, residual = 5.7194519497929444e-05     
CG iteration 7, residual = 3.864534071742561e-05     
CG iteration 8, residual = 3.073978871526682e-05     
CG iteration 9, residual = 2.3020823766624592e-05     
CG iteration 10, residual = 1.812595283960419e-05     
CG iteration 11, residual = 1.5009828463389428e-05     
CG iteration 12, residual = 9.929894730881364e-06     
CG iteration 13, residual = 7.676282770832387e-06     
CG iteration 14, residual = 6.010333033265084e-06     
CG iteration 15, residual = 4.187238350780369e-06     
CG iteration 16, residual = 3.292442097274219e-06     
CG iteration 17, residual = 2.762818338324048e-06     
CG iteration 18, residual = 2.0644301640377843e-06     
CG iteration 1

In [28]:
with TaskManager():
    fesstress = MatrixValued(H1(mesh,order=3), symmetric=True)
    gfstress = GridFunction(fesstress)
    gfstress.Interpolate (Stress(Sym(Grad(gfu))))

In [29]:
Draw (5e4*gfu, mesh, height="3vh");

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

In [30]:
Draw (Norm(gfstress), mesh, deformation=1e4*gfu, draw_vol=False, order=3, height="3vh");

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…